In [ ]:
"""
    Build modeling table
  - Join embedding CSVs with rq1_compare GPKGs on GEOID
  - Drop tracts missing hi_c
  - Compute hi_minus_lst and standardized hi_minus_lst_z (pooled)
  - Save tidy CSV to outputs/

To add a new city later: add it to CITIES dict below. Nothing else changes.
"""

import pandas as pd
import geopandas as gpd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"

# Extend this dict to add more cities later (just need paths to embeddings CSV and rq1_compare GPKG)
CITIES = {
    "houston": {
        "embeddings": DATA / "houston_embeddings.csv",
        "gpkg":       DATA / "houston_rq1_compare.gpkg",
    },
    "phoenix": {
        "embeddings": DATA / "phoenix_embeddings.csv",
        "gpkg":       DATA / "phoenix_rq1_compare.gpkg",
    },
}

EMBED_COLS = [f"A{i:02d}" for i in range(64)]   # A00–A63
DEMO_COLS  = ["poverty_rate", "median_household_income_clean",
              "low_income_flag", "high_poverty_flag", "inequity_group"]
HEAT_COLS  = ["lst_c", "hi_c", "compare_group", "lst_hotspot", "hi_hotspot"]

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────


def load_city(city_name: str, cfg: dict) -> pd.DataFrame:
    # 1. Read embedding CSV
    emb = pd.read_csv(cfg["embeddings"], dtype={"GEOID": str})

    # 2. Read only heat columns from GPKG (drop geometry — not needed for modeling)
    gpkg = gpd.read_file(cfg["gpkg"])[["GEOID"] + HEAT_COLS]
    gpkg["GEOID"] = gpkg["GEOID"].astype(str)

    # 3. Join on GEOID
    df = emb.merge(gpkg, on="GEOID", how="inner")

    # 4. Drop tracts missing hi_c (target can't be computed)
    n_before = len(df)
    df = df.dropna(subset=["hi_c"])
    n_dropped = n_before - len(df)
    print(f"  {city_name}: {n_before} tracts → dropped {n_dropped} missing hi_c → {len(df)} kept")

    # 5. Tag city (should already exist, but make explicit)
    df["city"] = city_name

    return df


def main():
    frames = []
    for city_name, cfg in CITIES.items():
        print(f"Loading {city_name}...")
        frames.append(load_city(city_name, cfg))

    df = pd.concat(frames, ignore_index=True)
    print(f"\nPooled sample: {len(df)} tracts")

    # 6. Compute target variable
    df["hi_minus_lst"] = df["hi_c"] - df["lst_c"]

    # Standardize across the POOLED sample (z-score)
    mu  = df["hi_minus_lst"].mean()
    std = df["hi_minus_lst"].std()
    df["hi_minus_lst_z"] = (df["hi_minus_lst"] - mu) / std
    print(f"hi_minus_lst  → mean={mu:.3f}, std={std:.3f}")
    print(f"hi_minus_lst_z→ mean≈0, std≈1 (check: {df['hi_minus_lst_z'].mean():.4f}, {df['hi_minus_lst_z'].std():.4f})")

    # 7. Save
    out_cols = ["GEOID", "city"] + EMBED_COLS + DEMO_COLS + HEAT_COLS + ["hi_minus_lst", "hi_minus_lst_z"]
    out = df[out_cols]

    out_path = ROOT / "outputs" / "modeling_table.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_path, index=False)
    print(f"\nSaved → {out_path}")
    print(f"Final shape: {out.shape}  (rows × cols)")

    # 8. Quick sanity: city breakdown
    print("\nCity breakdown:")
    print(out.groupby("city")["hi_minus_lst_z"].agg(["count", "mean", "std"]).round(3))


if __name__ == "__main__":
    main()